# Hourly PM2.5 forecast competition: seven methods on a time-shifted split

This page compares seven fitted methods on one forecasting task: predict hourly PM2.5 at a Beijing station for 2014 from weather covariates, after training on 2010–2013. The methods are mboostrs, XGBoost, LightGBM, CatBoost, pyGAM, statsmodels quantile regression on a spline basis, and a climatology floor. Each method was run by a separate analyst who saw only the task brief and their own library, so no method's settings were chosen with knowledge of another's results.

The methods were chosen for the problem. The target is positive and heavy-tailed (the 99th percentile is about 400 µg/m³ against a median near 70), the covariates include a categorical wind direction, the relation to weather is nonlinear and interacts with season and hour, and the test year sits in a different part of the covariate range than the training years. That mix favours tree boosters with native categorical handling (XGBoost, LightGBM, CatBoost), additive smooth models (mboostrs, pyGAM), and a classical quantile regression baseline (statsmodels). The floor shows what a seasonal climatology achieves with no covariates.

Every method predicts the 10th, 50th and 90th percentiles of PM2.5 for each 2014 hour. The scores below use the test labels, which the analysts did not see.

:::{note} Recorded before the response-units change
The mboostrs `Quantile` forecasts on this page are recorded results. They were recorded with mboostrs 0.2.3 or earlier, when distributional (LSS) and `Quantile` fits boosted the response in the units it was given. Since then these fits divide the response by its standard deviation, boost the rescaled response and map the fitted model back (`response_scale="auto"`), so a step has the same size in any units; `Quantile` also starts at the `tau`-quantile of the response, and takes steps of 0.3 when no control is given. Refitting on the current version gives different numbers. `response_scale=None` with the settings the page states reproduces the recorded fits; [Response units](../implementation/parity_and_bugs.ipynb#response-units) describes the change and its measured effect.
:::

## Task and split

The data is the Beijing PM2.5 hourly series in `docs/book/case_studies/beijing_pm25.csv`. Rows with a missing PM2.5 value were removed from training and from scoring. Training is 33,096 hourly rows from 2010–2013. The test set is 8,661 hourly rows from 2014. Test features contain no missing values. Target lags were not allowed as features, since the true PM2.5 of the previous hour is unknown at forecast time in 2014. Lags and rolling means of the weather covariates were allowed.

Tuning was done by each analyst on a time-ordered split inside the training years (fit 2010–2012, validate 2013). Each analyst then refit on all of 2010–2013 before predicting 2014. Tuning budgets were the same for all: at most 40 configurations and 30 minutes of fitting. The refit timings below are wall-clock seconds reported by each analyst's script, measured while other analysts were running on the same machine, so they are indicative only.

In [1]:
from pathlib import Path

import numpy as np
import polars as pl
from great_tables import GT, loc, style

ROOT = Path("pm25_forecast")
TRUTH = pl.read_parquet(ROOT / "data/test_truth.parquet")
FEATURES = pl.read_parquet(ROOT / "data/test_features.parquet").select("No", "month")
METHODS = {
    "mboostrs": "mboostrs (gamboost, Quantile family)",
    "xgboost": "XGBoost (reg:quantileerror)",
    "lightgbm": "LightGBM (quantile objective)",
    "catboost": "CatBoost (MultiQuantile)",
    "pygam": "pyGAM (Gaussian GAM on log scale + residual quantiles)",
    "statsmodels_qr": "statsmodels QuantReg (spline basis, log scale)",
    "climatology": "Climatology floor (month x hour empirical quantiles)",
}
QS = (0.1, 0.5, 0.9)

In [2]:
def pinball(y, q, tau):
    d = y - q
    return np.mean(np.maximum(tau * d, (tau - 1) * d))


def score(name):
    p = pl.read_parquet(ROOT / "runs" / name / "predictions.parquet")
    assert p.height == TRUTH.height, f"{name}: {p.height} rows"
    assert p["No"].n_unique() == TRUTH.height, f"{name}: duplicate No"
    j = TRUTH.join(FEATURES, on="No").join(p, on="No", how="left")
    assert j.null_count().sum_horizontal().item() == 0, f"{name}: missing predictions"
    y = j["pm2.5"].to_numpy()
    q10, q50, q90 = (j[c].to_numpy() for c in ("q10", "q50", "q90"))
    assert np.all(q10 <= q50 + 1e-9) and np.all(q50 <= q90 + 1e-9), f"{name}: quantiles cross"
    inside = (y >= q10) & (y <= q90)
    tail = y > 150
    winter = j["month"].is_in([11, 12, 1, 2, 3]).to_numpy()
    return {
        "method": name,
        "rmse_log1p": np.sqrt(np.mean((np.log1p(y) - np.log1p(q50)) ** 2)),
        "mae": np.mean(np.abs(y - q50)),
        "pinball": np.mean([pinball(y, q, t) for q, t in zip((q10, q50, q90), QS)]),
        "cov80": inside.mean(),
        "width80": np.mean(q90 - q10),
        "tail_mae": np.mean(np.abs(y[tail] - q50[tail])),
        "cov80_winter": inside[winter].mean(),
        "cov80_summer": inside[~winter].mean(),
    }


scores = pl.DataFrame([score(m) for m in METHODS]).sort("pinball")
scores

method,rmse_log1p,mae,pinball,cov80,width80,tail_mae,cov80_winter,cov80_summer
str,f64,f64,f64,f64,f64,f64,f64,f64
"""lightgbm""",0.621282,40.350197,13.084801,0.677635,103.967701,95.781447,0.617754,0.719646
"""xgboost""",0.622862,40.746746,13.179927,0.673132,101.473228,98.185487,0.631756,0.702161
"""catboost""",0.618072,40.85107,13.18112,0.679714,101.476865,97.075578,0.618874,0.722397
"""mboostrs""",0.661485,44.102711,14.145475,0.738483,137.754196,105.601315,0.679922,0.779568
"""pygam""",0.677369,45.247288,14.543945,0.761806,148.901703,108.999505,0.745729,0.773084
"""statsmodels_qr""",0.705431,47.26039,14.690428,0.743332,135.558355,114.010318,0.657799,0.80334
"""climatology""",1.052882,66.112343,20.387476,0.773698,204.595659,174.602763,0.718566,0.812377


In [3]:
# Best value per column is bolded. Coverage is best when closest to the nominal 0.80.
best_rows = {
    "rmse_log1p": scores["rmse_log1p"].arg_min(),
    "mae": scores["mae"].arg_min(),
    "pinball": scores["pinball"].arg_min(),
    "tail_mae": scores["tail_mae"].arg_min(),
    "width80": scores["width80"].arg_min(),
    "cov80": (scores["cov80"] - 0.80).abs().arg_min(),
}
table = (
    GT(scores.with_columns(pl.col("method").replace(METHODS)))
    .tab_header(
        title="Hourly PM2.5 forecasts for 2014, seven methods",
        subtitle="Lower is better for errors and width; 80% coverage should be near 0.80",
    )
    .tab_spanner(label="Median accuracy", columns=["rmse_log1p", "mae", "tail_mae"])
    .tab_spanner(label="Quantiles (10/50/90)", columns=["pinball", "cov80", "width80"])
    .tab_spanner(label="Coverage by season", columns=["cov80_winter", "cov80_summer"])
    .fmt_number(columns=["rmse_log1p", "pinball", "cov80", "cov80_winter", "cov80_summer"], decimals=3)
    .fmt_number(columns=["mae", "tail_mae", "width80"], decimals=1)
    .cols_label(
        method="Method",
        rmse_log1p="RMSE log1p",
        mae="MAE µg/m³",
        tail_mae="MAE, PM2.5 > 150",
        pinball="Mean pinball",
        cov80="Coverage",
        width80="Width µg/m³",
        cov80_winter="Winter",
        cov80_summer="Summer",
    )
    .tab_source_note("Winter is November to March. Pinball is averaged over the 0.1, 0.5 and 0.9 quantiles on the raw µg/m³ scale.")
)
for col, row in best_rows.items():
    table = table.tab_style(style.text(weight="bold"), loc.body(columns=col, rows=[int(row)]))
table

GT(_tbl_data=shape: (7, 9)
┌───────────┬───────────┬───────────┬───────────┬───┬───────────┬───────────┬───────────┬──────────┐
│ method    ┆ rmse_log1 ┆ mae       ┆ pinball   ┆ … ┆ width80   ┆ tail_mae  ┆ cov80_win ┆ cov80_su │
│ ---       ┆ p         ┆ ---       ┆ ---       ┆   ┆ ---       ┆ ---       ┆ ter       ┆ mmer     │
│ str       ┆ ---       ┆ f64       ┆ f64       ┆   ┆ f64       ┆ f64       ┆ ---       ┆ ---      │
│           ┆ f64       ┆           ┆           ┆   ┆           ┆           ┆ f64       ┆ f64      │
╞═══════════╪═══════════╪═══════════╪═══════════╪═══╪═══════════╪═══════════╪═══════════╪══════════╡
│ LightGBM  ┆ 0.621282  ┆ 40.350197 ┆ 13.084801 ┆ … ┆ 103.96770 ┆ 95.781447 ┆ 0.617754  ┆ 0.719646 │
│ (quantile ┆           ┆           ┆           ┆   ┆ 1         ┆           ┆           ┆          │
│ objective ┆           ┆           ┆           ┆   ┆           ┆           ┆           ┆          │
│ )         ┆           ┆           ┆           ┆   ┆           ┆           ┆           ┆          │
│ XGBoost   ┆ 0.622862  ┆ 40.746746 ┆ 13.179927 ┆ … ┆ 101.47322 ┆ 98.185487 ┆ 0.631756  ┆ 0.702161 │
│ (reg:quan ┆           ┆           ┆           ┆   ┆ 8         ┆           ┆           ┆          │
│ tileerror ┆           ┆           ┆           ┆   ┆           ┆           ┆           ┆          │
│ )         ┆           ┆           ┆           ┆   ┆           ┆           ┆           ┆          │
│ CatBoost  ┆ 0.618072  ┆ 40.85107  ┆ 13.18112  ┆ … ┆ 101.47686 ┆ 97.075578 ┆ 0.618874  ┆ 0.722397 │
│ (MultiQua ┆           ┆           ┆           ┆   ┆ 5         ┆           ┆           ┆          │
│ ntile)    ┆           ┆           ┆           ┆   ┆           ┆           ┆           ┆          │
│ mboostrs  ┆ 0.661485  ┆ 44.102711 ┆ 14.145475 ┆ … ┆ 137.75419 ┆ 105.60131 ┆ 0.679922  ┆ 0.779568 │
│ (gamboost ┆           ┆           ┆           ┆   ┆ 6         ┆ 5         ┆           ┆          │
│ ,         ┆           ┆           ┆           ┆   ┆           ┆           ┆           ┆          │
│ Quantile  ┆           ┆           ┆           ┆   ┆           ┆           ┆           ┆          │
│ f…        ┆           ┆           ┆           ┆   ┆           ┆           ┆           ┆          │
│ pyGAM     ┆ 0.677369  ┆ 45.247288 ┆ 14.543945 ┆ … ┆ 148.90170 ┆ 108.99950 ┆ 0.745729  ┆ 0.773084 │
│ (Gaussian ┆           ┆           ┆           ┆   ┆ 3         ┆ 5         ┆           ┆          │
│ GAM on    ┆           ┆           ┆           ┆   ┆           ┆           ┆           ┆          │
│ log sca…  ┆           ┆           ┆           ┆   ┆           ┆           ┆           ┆          │
│ statsmode ┆ 0.705431  ┆ 47.26039  ┆ 14.690428 ┆ … ┆ 135.55835 ┆ 114.01031 ┆ 0.657799  ┆ 0.80334  │
│ ls        ┆           ┆           ┆           ┆   ┆ 5         ┆ 8         ┆           ┆          │
│ QuantReg  ┆           ┆           ┆           ┆   ┆           ┆           ┆           ┆          │
│ (spline   ┆           ┆           ┆           ┆   ┆           ┆           ┆           ┆          │
│ b…        ┆           ┆           ┆           ┆   ┆           ┆           ┆           ┆          │
│ Climatolo ┆ 1.052882  ┆ 66.112343 ┆ 20.387476 ┆ … ┆ 204.59565 ┆ 174.60276 ┆ 0.718566  ┆ 0.812377 │
│ gy floor  ┆           ┆           ┆           ┆   ┆ 9         ┆ 3         ┆           ┆          │
│ (month x  ┆           ┆           ┆           ┆   ┆           ┆           ┆           ┆          │
│ hou…      ┆           ┆           ┆           ┆   ┆           ┆           ┆           ┆          │
└───────────┴───────────┴───────────┴───────────┴───┴───────────┴───────────┴───────────┴──────────┘, _body=<great_tables._gt_data.Body object at 0x10b5ddeb0>, _boxhead=Boxhead([ColInfo(var='method', type=<ColInfoTypeEnum.default: 1>, column_label='Method', column_align='left', column_width=None), ColInfo(var='rmse_log1p', type=<ColInfoTypeEnum.default: 1>, column_label='RMSE log1p', column_align='right', column_width=None), ColInfo(v

## Method settings and tuning

The table gives the final settings. Each method's tuning grid and refit time are in `pm25_forecast/runs/<method>/REPORT.md`, and the configuration history is in `tuning_log.csv`.

| Method | Final model | Tuning | Refit (s) |
|---|---|---|---|
| mboostrs | Three `gamboost` fits, one per quantile (Quantile family), on log1p(PM2.5). `bbs` terms on hour, month, TEMP, PRES, DEWP, Iws, day of year and 24-hour rolling means, each `df=4`; `bols` for wind direction, snow and rain; `nu=0.1` (the step length). mstop (the number of boosting steps) 6000 (q10), 4800 (q50), 3200 (q90). | 25 configurations, 265 s summed fit time | 52.8 (includes predict) |
| XGBoost | `reg:quantileerror` with three quantiles, depth 4, learning rate 0.1, 775 rounds | 12 configurations, 136 s | 8.6 |
| LightGBM | Three `quantile` models, 15 leaves, learning rate 0.03, 550 rounds | 24 configurations, 403 s | 3.1 |
| CatBoost | `MultiQuantile` loss, depth 3, 824 iterations, wind direction as a native categorical | 16 configurations, 104 s | 6.0 |
| pyGAM | Gaussian GAM on log1p(PM2.5) with smooth terms and a tensor on hour and month; q10 and q90 are the median plus residual quantiles from a 2013 holdout | 8 configurations, 47 s | 8.8 |
| statsmodels QuantReg | Three `QuantReg` fits on a spline basis with winter interactions, log1p target | 18 configurations, 797 s | 12.5 |
| Climatology floor | Empirical month-by-hour quantiles from 2010–2013 | none | n/a |

The climatology floor has no covariates. It is the reference that every method should beat.

### Friction log

Each analyst listed the points where the library made the task harder. These are the library-level findings the page can support.

In [4]:
friction = pl.DataFrame(
    {
        "method": [
            "mboostrs", "mboostrs", "mboostrs",
            "XGBoost", "XGBoost", "XGBoost",
            "LightGBM", "LightGBM", "LightGBM",
            "CatBoost", "CatBoost", "CatBoost",
            "pyGAM", "pyGAM", "pyGAM",
            "statsmodels QuantReg", "statsmodels QuantReg", "statsmodels QuantReg",
        ],
        "point": [
            "Long paths warn that boosting has undone much of the penalty; the df setting and the tuned mstop interact.",
            "No time-ordered helper for a quantile loss; the mstop path loop was written by hand. cvrisk with custom folds was not tried.",
            "Out-of-range covariates warn at predict time (DEWP below the training range) rather than being clipped; the analyst ran with warnings suppressed.",
            "No original-scale pinball metric or early stopping for the log-scale objective; round selection needed a checkpoint loop.",
            "Multi-quantile output comes from independent trees, so quantiles can cross; sorting was done in post-processing.",
            "Categorical features need enable_categorical and pandas category dtype; the 1000-round cap was reached by two low-learning-rate configurations.",
            "One alpha per model, so three training runs; crossing can occur. Round selection needed a loop over predict(num_iteration=k).",
            "No early stopping on the quantile metric without a validation set.",
            "Categorical features need pandas category dtype with explicit levels; the Polars frame was converted column by column.",
            "Early stopping cannot target raw-scale pinball; the only in-training metric is MultiQuantile on log1p.",
            "MultiQuantile has no ordering guarantee; rows were sorted after prediction, crossings not counted.",
            "cat_features needed a pandas frame with cbwd as str; no Polars input path was used.",
            "No quantile or distributional output; residual quantiles are the only route, and the offsets do not vary with covariates.",
            "gridsearch with a one-dimensional lam array applies one value to all terms; GCV uses training data only, so there is no validation hook.",
            "f() terms need integer codes; a string categorical had to be mapped by hand.",
            "patsy bs() raises NotImplementedError for points outside the outer knots (validation and test rows).",
            "Winter-interaction columns plus month dummies are aliased; QuantReg hit max_iter=5000 (about 100 s per fit) until columns were pruned by pivoted QR.",
            "patsy.dmatrix with an outcome term fails when the design is built for prediction; the design must use the right-hand side only.",
        ],
    }
)
friction
friction

method,point
str,str
"""mboostrs""","""Long paths warn that boosting …"
"""mboostrs""","""No time-ordered helper for a q…"
"""mboostrs""","""Out-of-range covariates warn a…"
"""XGBoost""","""No original-scale pinball metr…"
"""XGBoost""","""Multi-quantile output comes fr…"
…,…
"""pyGAM""","""gridsearch with a one-dimensio…"
"""pyGAM""","""f() terms need integer codes; …"
"""statsmodels QuantReg""","""patsy bs() raises NotImplement…"


## Scorecard

The knowledge-discovery scorecard in `docs/scorecard/scorecard.md` grades a library on ten dimensions, each with an anchored 1–5 rubric. Each rubric names its own measurement, and most of those measurements need a planted-truth suite, a standard timing workload, or a human or agent study. This battery measures two of them directly, D5 (iteration speed) and D9 (prediction-interval calibration). The table scores those two per library and gives the reason for each other dimension. No weighted aggregate is computed, because the scorecard asks for the per-dimension profile and eight of ten dimensions are unscored here.

In [5]:
LIBS = ["mboostrs", "XGBoost", "LightGBM", "CatBoost", "pyGAM", "statsmodels QuantReg"]
sheet = pl.DataFrame(
    {
        "Dimension": [
            "D1 Relationship discovery",
            "D2 Causal humility and support",
            "D3 Explanation quality",
            "D4 Importance quantification",
            "D5 Iteration speed (proxy)",
            "D6 Robustness across conditions",
            "D7 Human-friendliness",
            "D8 Agent-friendliness",
            "D9 Uncertainty honesty",
            "D10 Reproducibility",
        ],
        "mboostrs": ["not scored", "not scored", "not scored", "not scored", "2.5", "not scored", "not scored", "not scored", "2", "not scored"],
        "XGBoost": ["not scored", "not scored", "not scored", "not scored", "2.5", "not scored", "not scored", "not scored", "2", "not scored"],
        "LightGBM": ["not scored", "not scored", "not scored", "not scored", "2", "not scored", "not scored", "not scored", "2", "not scored"],
        "CatBoost": ["not scored", "not scored", "not scored", "not scored", "3", "not scored", "not scored", "not scored", "2", "not scored"],
        "pyGAM": ["not scored", "not scored", "not scored", "not scored", "3", "not scored", "not scored", "not scored", "2", "not scored"],
        "statsmodels QuantReg": ["not scored", "not scored", "not scored", "not scored", "2", "not scored", "not scored", "not scored", "2", "not scored"],
        "Evidence": [
            "Needs planted-truth data with known structure (TPR and FDR). This task has no known structure.",
            "Documentation audit not run.",
            "Narrative report task not run. The REPORT.md files are not graded for narrative.",
            "Effect-size intervals not evaluated. The coverage here is for predictive intervals (D9).",
            "Mean tuning seconds per configuration: pyGAM 6, CatBoost 6.5, mboostrs 10.6, XGBoost 11, LightGBM 17, statsmodels 44. The standard 50k x 20 follow-up workload was not run.",
            "All six ran through the 2014 shift. statsmodels needed a patsy bounds fix. Small-n, messy and large-n conditions were not run.",
            "Needs a task-based study or issue-tracker analysis. Not run.",
            "Needs the frozen-prompt agent protocol. The seven runs used the same brief. Three REPORT.md writes were refused by the agent harness, which is an infrastructure limit and not a library property.",
            "Intervals widen with the predicted level for every library, so adaptive widening is present. No library's 80% band is approximately nominal in any season or level group (cell coverage 0.62–0.80). No library warns when its band is unreliable.",
            "Each run has run.py, a tuning log and fixed seeds. One-command reproduction was not run.",
        ],
    }
)
sheet

Dimension,mboostrs,XGBoost,LightGBM,CatBoost,pyGAM,statsmodels QuantReg,Evidence
str,str,str,str,str,str,str,str
"""D1 Relationship discovery""","""not scored""","""not scored""","""not scored""","""not scored""","""not scored""","""not scored""","""Needs planted-truth data with …"
"""D2 Causal humility and support""","""not scored""","""not scored""","""not scored""","""not scored""","""not scored""","""not scored""","""Documentation audit not run."""
"""D3 Explanation quality""","""not scored""","""not scored""","""not scored""","""not scored""","""not scored""","""not scored""","""Narrative report task not run.…"
"""D4 Importance quantification""","""not scored""","""not scored""","""not scored""","""not scored""","""not scored""","""not scored""","""Effect-size intervals not eval…"
"""D5 Iteration speed (proxy)""","""2.5""","""2.5""","""2""","""3""","""3""","""2""","""Mean tuning seconds per config…"
"""D6 Robustness across condition…","""not scored""","""not scored""","""not scored""","""not scored""","""not scored""","""not scored""","""All six ran through the 2014 s…"
"""D7 Human-friendliness""","""not scored""","""not scored""","""not scored""","""not scored""","""not scored""","""not scored""","""Needs a task-based study or is…"
"""D8 Agent-friendliness""","""not scored""","""not scored""","""not scored""","""not scored""","""not scored""","""not scored""","""Needs the frozen-prompt agent …"
"""D9 Uncertainty honesty""","""2""","""2""","""2""","""2""","""2""","""2""","""Intervals widen with the predi…"


### D5 and D9 scoring

**D5** uses the rubric anchors for time to answer. The score is the reported tuning-session time divided by the number of configurations, which is a per-configuration figure. A score of 3 means seconds per fit, 2 means tens of seconds, and half-points sit between the two. The standard workload in the rubric is not run here, so this is a proxy.

**D9** is scored 2 for every library. The rubric's score of 3 needs coverage that is approximately nominal on standard data, and none of these libraries reach that. The rubric's score of 4 needs adaptive widening, which is present, but the rubric's levels are cumulative, so the 4 anchor also requires calibration.

In [6]:
# Width and coverage in each season and predicted-level cell, from the same predictions.
def cells(name):
    p = pl.read_parquet(ROOT / "runs" / name / "predictions.parquet")
    j = TRUTH.join(FEATURES.select("No", "month"), on="No").join(p, on="No").with_columns(
        width=pl.col("q90") - pl.col("q10"),
        winter=pl.col("month").is_in([11, 12, 1, 2, 3]),
        inside=(pl.col("pm2.5") >= pl.col("q10")) & (pl.col("pm2.5") <= pl.col("q90")),
        high=pl.col("q50") > 100,
    )
    season = j.group_by("winter").agg(pl.col("width").mean(), pl.col("inside").mean())
    level = j.group_by("high").agg(pl.col("width").mean(), pl.col("inside").mean())
    row = {"method": name}
    for key, label in ((False, "summer"), (True, "winter")):
        r = season.filter(pl.col("winter") == key).row(0, named=True)
        row[f"{label} width"], row[f"{label} cov"] = r["width"], r["inside"]
    for key, label in ((False, "low (median <= 100)"), (True, "high (median > 100)")):
        r = level.filter(pl.col("high") == key).row(0, named=True)
        row[f"{label} width"], row[f"{label} cov"] = r["width"], r["inside"]
    return row


cell_table = pl.DataFrame([cells(m) for m in ["mboostrs", "xgboost", "lightgbm", "catboost", "pygam", "statsmodels_qr"]])
cell_table

method,summer width,summer cov,winter width,winter cov,low (median <= 100) width,low (median <= 100) cov,high (median > 100) width,high (median > 100) cov
str,f64,f64,f64,f64,f64,f64,f64,f64
"""mboostrs""",133.769545,0.779568,143.433803,0.679922,97.227134,0.723558,209.36927,0.764856
"""xgboost""",98.484329,0.702161,105.733528,0.631756,73.187691,0.672952,155.230911,0.673476
"""lightgbm""",101.957943,0.719646,106.83235,0.617754,73.541826,0.66397,158.575225,0.702162
"""catboost""",98.580864,0.722397,105.604741,0.618874,69.868963,0.679964,158.291457,0.679264
"""pygam""",147.805578,0.773084,150.464088,0.745729,91.412094,0.744699,259.969136,0.794854
"""statsmodels_qr""",130.342094,0.80334,142.993463,0.657799,92.41478,0.736349,221.621731,0.757261


## Results by dimension

**D9, uncertainty honesty.** All six libraries widen their 80% bands with the predicted level: the low-level bands are 70–100 µg/m³ and the high-level bands are 155–260 µg/m³. That is the adaptive behaviour the rubric asks for. The coverage, though, is below the nominal 0.80 in most cells. The best cells are pyGAM and mboostrs in the high-level group (hours grouped by predicted PM2.5 level), at 0.80 and 0.76. The worst are the tree boosters in winter, at 0.62–0.63. Because no library's coverage is approximately nominal in every cell, each scores 2.

**D5, iteration speed (proxy).** CatBoost and pyGAM take about 6 seconds per configuration, which is seconds per fit. mboostrs and XGBoost take about 11 seconds, and LightGBM about 17. statsmodels takes about 44 seconds, and its single refit took 12.5 seconds. The speed ranking here is one measure of cost, not the workload the rubric specifies.

**Dimensions not scored.** D1 through D4 need planted-truth data, a documentation audit, and a report task. D6 and D7 need the small-n, messy and large-n conditions and a human or issue-tracker study. D8 needs the frozen-prompt agent protocol. D10 needs an independent reproduction. None of these was run on this battery.

## Opportunities for mboostrs, by dimension

The benchmark points to four changes in mboostrs. Each one maps to a rubric dimension.

1. **D9, calibration.** The 80% band covers 74% overall, with 68% in winter. The three quantile fits use separate mstop values, chosen by pinball loss (the quantile-regression loss at level τ) on one 2013 holdout. The q10 optimum sits at the end of the 6000-iteration path, so that choice is not settled. A time-ordered cross-validation helper for the quantile family would test whether the stopping rule is the cause. This is the change most likely to move the mboostrs score from 2 toward 3.

2. **D9, the anchor 5 diagnostic.** No library warns when its band is unreliable. mboostrs could flag predicted levels or seasons where observed coverage in holdout falls below the nominal level.

3. **D6, shift diagnostics.** At predict time mboostrs warns about DEWP values below the training range, but it does not report how many rows are affected or the fraction per term. The analyst ran with warnings suppressed. Rubric anchor 5, the written description of what the top grade requires, asks for warnings when findings may not transfer, so a summary of out-of-range counts per term is the specific gap.

4. **D5 and D7, long paths.** Fits past 3000 iterations warn that boosting has undone much of the penalty. The warning names a term that has spent most of its 4 degrees of freedom (`bbs(Iws_r24)`: 11.5 of 24). Long paths and a fixed df interact here. A df-aware stopping rule would remove the warning and the cost of the long path together.

## Caveats

- One analyst ran each library, and the analysts differed in their familiarity with the library. A stronger analyst could move a library's numbers. The mboostrs analyst is not the library's maintainer.
- CatBoost's tuning score used early stopping on the 2013 split, so its tuning score is optimistic. Its test scores are not affected by that choice.
- The refit and tuning times were measured while several analysts shared the machine. They are indicative only, and the D5 scores are a proxy for the rubric's standard workload.
- statsmodels' tuning fits hit the iteration limit before the aliased columns were removed. The final run used the pruned design, and the reported numbers are from that run.
- Only one station and one test year are used. The coverage cells rest on about 8,700 hourly rows from one year, and those rows are strongly autocorrelated, so the coverage differences between libraries are not a settled ranking.
- The test labels are published here, so this split cannot be reused as a fresh holdout for these methods.
- The scorecard in `docs/scorecard/` is an instrument, not a verdict. Its planted-truth battery is future work, so eight of its ten dimensions are unscored on this page.